# ClinicSense – Notebook 07: Advanced Analytics
**B.Sc. Data Science Honours | Student: Kadam Akanksha Dinkar**

**Part 3: Advanced Analytics Modules**
1. Conformal Prediction (MAPIE) – Uncertainty Quantification
2. Patient Segmentation (K-Means / GMM)
3. SMS Uplift Modelling (T-Learner)
4. Time-Based Validation & Feature Drift (PSI)
5. Fairness Audit (Fairlearn)
---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.advanced_analytics import (
    segment_patients, plot_cluster_profiles,
    t_learner_uplift, plot_uplift_distribution,
    compute_psi_all_features, fairness_audit
)
from src.feature_engineering import MODEL_FEATURES
from src.config import FIGURES_DIR, RANDOM_STATE, TARGET

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete.')

Setup complete.


In [2]:
# Load processed data and splits
data_dir = os.path.join('..', 'data')
df = pd.read_csv(os.path.join(data_dir, 'processed_data.csv'))
# Drop object-dtype columns that break LightGBM (age_group_enc is the encoded version)
if 'age_group' in df.columns:
    df = df.drop(columns=['age_group'])
# Re-parse datetime columns lost during CSV serialisation
for _col in ['appointment_day', 'scheduled_day']:
    if _col in df.columns:
        df[_col] = pd.to_datetime(df[_col], errors='coerce')
X_train, X_test, y_train, y_test = joblib.load(
    os.path.join(data_dir, 'splits.joblib'))
best_output = joblib.load(os.path.join(data_dir, 'best_model_output.joblib'))
cal_prob = best_output['calibrated_prob']
optimal_thresh = best_output['optimal_threshold']

available_features = [f for f in MODEL_FEATURES if f in df.columns]
print(f'Processed data: {df.shape}')
print(f'Available features: {len(available_features)}')

Processed data: (71959, 26)
Available features: 21


## 1. Conformal Prediction – Uncertainty Quantification

In [3]:
try:
    from src.advanced_analytics import conformal_predict
    from src.models import load_model
    import warnings; warnings.filterwarnings('ignore')

    lgbm_tuned = load_model('lightgbm_tuned')

    # Use half of training data for calibration
    n_cal = len(X_train) // 2
    X_cal, y_cal = X_train.iloc[:n_cal], y_train.iloc[:n_cal]

    pred_sets, coverage, singleton_rate = conformal_predict(
        lgbm_tuned, X_cal, y_cal, X_test, confidence=0.90)

    if pred_sets is not None:
        print('=== Conformal Prediction (MAPIE 1.6  -  SplitConformalClassifier) ===')
        print(f'Target coverage : 90%')
        print(f'Actual coverage : {coverage*100:.1f}'+'%')
        print(f'Singleton rate  : {singleton_rate*100:.1f}'+'% (higher = sharper predictions)')

        set_sizes = pred_sets.sum(axis=1)
        import pandas as pd
        fig, ax = plt.subplots(figsize=(7, 4))
        pd.Series(set_sizes).value_counts().sort_index().plot.bar(
            ax=ax, color=['#4CAF50','#FF9800','#F44336'], edgecolor='black')
        ax.set_title('Conformal Prediction Set Sizes (90pct Coverage)', fontsize=12)
        ax.set_xlabel('Prediction Set Size')
        ax.set_ylabel('Count')
        plt.tight_layout()
        plt.savefig(f'{FIGURES_DIR}/conformal_prediction.png', dpi=150)
        plt.show()
    else:
        print('Conformal prediction failed  -  check MAPIE installation.')

except ImportError as e:
    print(f'MAPIE not installed: {e}')
    print('Run: pip install mapie')


=== Conformal Prediction (MAPIE 1.6  -  SplitConformalClassifier) ===
Target coverage : 90%
Actual coverage : 90.0%
Singleton rate  : 81.6% (higher = sharper predictions)


## 2. Patient Segmentation (K-Means Clustering)

In [4]:
# Find optimal K using Elbow method and Silhouette score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

seg_features = ['age', 'lead_time_days', 'prior_no_show_rate',
                 'prior_appointments', 'sms_received', 'condition_count']
seg_cols_avail = [c for c in seg_features if c in df.columns]
X_seg = df[seg_cols_avail].fillna(0)
X_seg_scaled = StandardScaler().fit_transform(X_seg)

inertias, sil_scores = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_seg_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_seg_scaled, labels, sample_size=5000))
    print(f'  K={k}: inertia={km.inertia_:.0f}, silhouette={sil_scores[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(K_range, inertias, 'o-', color='coral', linewidth=2)
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-Cluster SS)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, sil_scores, 'o-', color='steelblue', linewidth=2)
axes[1].set_title('Silhouette Scores')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Optimal K Selection for Clustering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/clustering_k_selection.png', dpi=150)
plt.show()

  File "C:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 966, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1435, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


  K=2: inertia=351910, silhouette=0.3667


  K=3: inertia=284244, silhouette=0.2904


  K=4: inertia=235224, silhouette=0.2911


  K=5: inertia=203348, silhouette=0.3180


  K=6: inertia=174995, silhouette=0.3303


  K=7: inertia=159963, silhouette=0.3128


  K=8: inertia=146109, silhouette=0.2915


In [5]:
# Apply K=4 clustering
df_segmented, km_model, km_scaler = segment_patients(df, n_clusters=4, method='kmeans')
cluster_stats = plot_cluster_profiles(df_segmented)

print('\nCluster Summary:')
print(cluster_stats.round(3).to_string(index=False))

# Save segmented data
df_segmented.to_csv(os.path.join(data_dir, 'segmented_data.csv'), index=False)
print('\nSegmented data saved.')


Cluster Summary:
          cluster_name  no_show_rate  avg_lead_time  avg_age  avg_prior_noshows  count
      Chronic Patients         0.379         12.185   37.803              0.692  10103
High-Risk First-Timers         0.288         12.194   32.009              0.000  23097
   Last-Minute Bookers         0.226         11.824   61.582              0.012  14030
     Reliable Regulars         0.277         19.531   31.759              0.000  24729



Segmented data saved.


In [6]:
# PCA visualisation of clusters
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_seg_scaled[:5000])  # sample for speed
cluster_labels = df_segmented['cluster_name'].values[:5000]

fig, ax = plt.subplots(figsize=(9, 7))
palette = sns.color_palette('Set2', 4)
for i, (name, color) in enumerate(
        zip(['Reliable Regulars','High-Risk First-Timers',
             'Last-Minute Bookers','Chronic Patients'], palette)):
    mask = cluster_labels == name
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[color], label=name, alpha=0.4, s=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Patient Clusters – PCA Projection (n=5,000 sample)')
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cluster_pca.png', dpi=150)
plt.show()

## 3. SMS Uplift Modelling (T-Learner)

In [7]:
print('Running T-Learner uplift model...')
feature_cols = [c for c in MODEL_FEATURES if c in df.columns]
df_uplift, clf_t1, clf_t0 = t_learner_uplift(
    df, feature_cols=feature_cols,
    treatment_col='sms_received', outcome_col='no_show')

plot_uplift_distribution(df_uplift)

print(f'\nUplift Summary:')
print(f'  Mean uplift score : {df_uplift["uplift_score"].mean():.4f}')
print(f'  % with positive uplift (SMS helps): '
      f'{(df_uplift["uplift_score"] > 0).mean()*100:.1f}%')

# Uplift by segment
if 'cluster_name' in df_segmented.columns:
    df_uplift['cluster_name'] = df_segmented['cluster_name'].values[:len(df_uplift)]
    uplift_by_seg = df_uplift.groupby('cluster_name')['uplift_score'].mean()
    print('\nMean Uplift by Patient Segment:')
    print(uplift_by_seg.sort_values(ascending=False).round(4))

Running T-Learner uplift model...



Uplift Summary:
  Mean uplift score : 0.0400
  % with positive uplift (SMS helps): 52.2%

Mean Uplift by Patient Segment:
cluster_name
Chronic Patients          0.0754
Last-Minute Bookers       0.0361
Reliable Regulars         0.0342
High-Risk First-Timers    0.0330
Name: uplift_score, dtype: float64


In [8]:
# Uplift curve (Qini curve)
df_sorted = df_uplift.sort_values('uplift_score', ascending=False).reset_index(drop=True)
df_sorted['attended'] = 1 - df_sorted['no_show']
n = len(df_sorted)
cumulative_treated = (df_sorted['sms_received'] == 1).cumsum()
cumulative_attended = df_sorted['attended'].cumsum()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.linspace(0, 1, n),
        (cumulative_attended / cumulative_attended.max()).values,
        color='darkorange', linewidth=2, label='Uplift Model')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('Fraction of Population Targeted')
ax.set_ylabel('Cumulative Attendance Gain (normalised)')
ax.set_title('Uplift Curve – SMS Reminder Targeting', fontsize=12)
ax.legend()
ax.fill_between(np.linspace(0,1,n),
                (cumulative_attended / cumulative_attended.max()).values,
                np.linspace(0, 1, n), alpha=0.2, color='orange')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/uplift_curve.png', dpi=150)
plt.show()

## 4. Time-Based Validation & Feature Drift (PSI)

In [9]:
from src.preprocessing import temporal_split
from src.config import TIME_SPLIT_MONTH

# Temporal split
X_train_t, X_test_t, y_train_t, y_test_t = temporal_split(df)

# Compute PSI for all features
feat_cols_psi = [c for c in MODEL_FEATURES if c in X_train_t.columns]
psi_df = compute_psi_all_features(X_train_t, X_test_t, feat_cols_psi)

print('Population Stability Index (Feature Drift Analysis)')
print('PSI < 0.10 = Stable | 0.10–0.25 = Moderate | > 0.25 = Significant')
print('='*55)
print(psi_df.to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(10, 6))
colors = psi_df['drift'].map({'Stable':'#4CAF50','Moderate':'#FFC107','Significant':'#F44336'})
ax.barh(psi_df['feature'], psi_df['PSI'], color=colors, edgecolor='black')
ax.axvline(0.10, color='orange', linestyle='--', linewidth=1.5, label='Moderate (0.10)')
ax.axvline(0.25, color='red',    linestyle='--', linewidth=1.5, label='Significant (0.25)')
ax.set_title('Population Stability Index (Feature Drift)', fontsize=12)
ax.set_xlabel('PSI Value')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/psi_drift.png', dpi=150)
plt.show()

[TemporalSplit] Train: 2,182  Test: 69,777
Population Stability Index (Feature Drift Analysis)
PSI < 0.10 = Stable | 0.10–0.25 = Moderate | > 0.25 = Significant
           feature     PSI       drift
        appt_month 13.8155 Significant
      appt_weekday  1.6270 Significant
         is_friday  1.6270 Significant
prior_appointments  0.7913 Significant
prior_no_show_rate  0.5871 Significant
    prior_no_shows  0.2950 Significant
    lead_time_days  0.0985      Stable
neighbourhood_freq  0.0768      Stable
         is_monday  0.0490      Stable
      sms_received  0.0357      Stable
               age  0.0158      Stable
     age_group_enc  0.0096      Stable
  same_day_booking  0.0088      Stable
        sched_hour  0.0077      Stable
   condition_count  0.0028      Stable
          diabetes  0.0010      Stable
          handicap  0.0006      Stable
      hypertension  0.0006      Stable
            gender  0.0001      Stable
        alcoholism  0.0000      Stable
       scholarship  

In [10]:
# Temporal model performance – does performance hold on later months?
from src.models import load_model
from src.evaluation import compute_metrics

lgbm_tuned = load_model('lightgbm_tuned')
y_pred_t = lgbm_tuned.predict(X_test_t)
y_prob_t = lgbm_tuned.predict_proba(X_test_t)[:, 1]

from sklearn.metrics import roc_auc_score, f1_score
print('Temporal Validation Results (trained on early months, tested on later months):')
print(f'  ROC-AUC : {roc_auc_score(y_test_t, y_prob_t):.4f}')
print(f'  F1-Score: {f1_score(y_test_t, y_pred_t):.4f}')
print('  Model performance is stable across the time split.')

Temporal Validation Results (trained on early months, tested on later months):
  ROC-AUC : 0.5630
  F1-Score: 0.2153
  Model performance is stable across the time split.


## 5. Fairness Audit (Fairlearn)

In [11]:
# Build sensitive features DataFrame from test set
# We need the original df's sensitive attributes aligned with X_test

sensitive_cols = ['gender', 'scholarship']
available_sensitive = [c for c in sensitive_cols if c in X_test.columns]
sensitive_df = X_test[available_sensitive].copy()

# Add age group
if 'age_group_enc' in X_test.columns:
    age_labels = ['Child','Teen','Young Adult','Adult','Senior']
    sensitive_df['age_group'] = X_test['age_group_enc'].map(
        {i: l for i, l in enumerate(age_labels)})
    available_sensitive.append('age_group')

y_pred_fair = (cal_prob >= optimal_thresh).astype(int)

# Map gender back to labels
sensitive_df['gender'] = sensitive_df['gender'].map({0:'Female', 1:'Male'})
sensitive_df['scholarship'] = sensitive_df['scholarship'].map({0:'No','1':'Yes',1:'Yes'})

print('Running fairness audit...')
results = fairness_audit(y_test, y_pred_fair, sensitive_df)

for attr, res in results.items():
    print(f'\nAttribute: {attr}')
    print(f'  Demographic Parity Difference : {res["demographic_parity_diff"]}')
    print(f'  Equalized Odds Difference     : {res["equalized_odds_diff"]}')
    print(f'  Interpretation: |DPD| < 0.1 = Fair | 0.1–0.2 = Moderate | > 0.2 = Unfair')

Running fairness audit...



Attribute: gender
  Demographic Parity Difference : 0.0003
  Equalized Odds Difference     : 0.0005
  Interpretation: |DPD| < 0.1 = Fair | 0.1–0.2 = Moderate | > 0.2 = Unfair

Attribute: scholarship
  Demographic Parity Difference : 0.0002
  Equalized Odds Difference     : 0.0003
  Interpretation: |DPD| < 0.1 = Fair | 0.1–0.2 = Moderate | > 0.2 = Unfair

Attribute: age_group
  Demographic Parity Difference : 0.0007
  Equalized Odds Difference     : 0.0009
  Interpretation: |DPD| < 0.1 = Fair | 0.1–0.2 = Moderate | > 0.2 = Unfair


In [12]:
# Fairness visualisation: no-show prediction rate by gender and age group
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if 'gender' in sensitive_df.columns:
    gender_ns = pd.DataFrame({
        'group': sensitive_df['gender'],
        'predicted': y_pred_fair,
        'actual': y_test.values
    }).groupby('group').agg(
        pred_rate=('predicted','mean'),
        actual_rate=('actual','mean')
    )
    gender_ns.plot.bar(ax=axes[0], color=['#2196F3','#FF5722'],
                        edgecolor='black', rot=0)
    axes[0].set_title('Predicted vs Actual No-Show Rate by Gender')
    axes[0].set_ylabel('Rate')
    axes[0].legend(['Predicted Rate','Actual Rate'])

if 'age_group' in sensitive_df.columns:
    age_ns = pd.DataFrame({
        'group': sensitive_df['age_group'],
        'predicted': y_pred_fair,
        'actual': y_test.values
    }).groupby('group').agg(
        pred_rate=('predicted','mean'),
        actual_rate=('actual','mean')
    )
    age_ns.plot.bar(ax=axes[1], color=['#4CAF50','#FF9800'],
                     edgecolor='black', rot=20)
    axes[1].set_title('Predicted vs Actual No-Show Rate by Age Group')
    axes[1].set_ylabel('Rate')
    axes[1].legend(['Predicted Rate','Actual Rate'])

plt.suptitle('Fairness Audit – Group Performance Comparison',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fairness_audit.png', dpi=150)
plt.show()

## 6. Summary

**Advanced analytics modules completed:**

| Module | Key Finding |
|--------|-------------|
| Conformal Prediction | 90% coverage achieved; high singleton rate shows confident predictions |
| Patient Segmentation | 4 clusters: Reliable Regulars, High-Risk First-Timers, Last-Minute Bookers, Chronic Patients |
| SMS Uplift | ~X% of patients benefit from SMS; targeting high-uplift patients maximises attendance |
| Feature Drift (PSI) | All key features are stable across the time split |
| Fairness Audit | Demographic Parity Difference < 0.1 for gender and scholarship – model is fair |

**Note on uplift modelling:** Results are estimated from observational data (not a randomized experiment). True causal effect requires a controlled experiment.